# Groq RAG with Local Embeddings

This notebook uses the local `all-MiniLM-L6-v2` embedding model and Groq for answer generation.

In [ ]:
import json
import os
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

current_directory = Path.cwd()
PROJECT_DIRECTORY = (
    current_directory / 'rag-for-beginners'
    if (current_directory / 'rag-for-beginners' / 'docs').is_dir()
    else current_directory
)
PDF_PATH = PROJECT_DIRECTORY / 'docs' / '66---16-06-2021-02-48-53.pdf'
PERSIST_DIRECTORY = PROJECT_DIRECTORY / 'db' / 'chroma_pdf_groq_local_embeddings'
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
GROQ_MODEL = 'openai/gpt-oss-120b'

## Load and split documents

In [ ]:
if not PDF_PATH.is_file():
    raise FileNotFoundError(f'PDF file not found: {PDF_PATH}')

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()
if not documents:
    raise FileNotFoundError(f'No pages found in {PDF_PATH}')

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f'Loaded {len(documents)} PDF pages and created {len(chunks)} chunks.')

Created a chunk of size 1029, which is longer than the specified 1000
Created a chunk of size 1206, which is longer than the specified 1000
Created a chunk of size 1055, which is longer than the specified 1000
Created a chunk of size 1082, which is longer than the specified 1000
Created a chunk of size 1027, which is longer than the specified 1000
Created a chunk of size 1187, which is longer than the specified 1000
Created a chunk of size 1518, which is longer than the specified 1000
Created a chunk of size 1603, which is longer than the specified 1000
Created a chunk of size 1436, which is longer than the specified 1000
Created a chunk of size 1039, which is longer than the specified 1000
Created a chunk of size 1078, which is longer than the specified 1000
Created a chunk of size 1043, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1068, which is longer than the specified 1000
Created a chunk of s

Loaded 5 documents and created 1330 chunks.


## Create or load the local vector store

In [4]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if PERSIST_DIRECTORY.is_dir():
    vector_store = Chroma(
        persist_directory=str(PERSIST_DIRECTORY),
        embedding_function=embeddings,
        collection_metadata={'hnsw:space': 'cosine'},
    )
else:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(PERSIST_DIRECTORY),
        collection_metadata={'hnsw:space': 'cosine'},
    )

print(f'Vector store ready at {PERSIST_DIRECTORY}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11434.37it/s]


Vector store ready at /Users/ankitsharma/Documents/Euron/Sachin Mishra/GenAI/rag-for-beginners/db/chroma_groq_local_embeddings


## Ask Groq using retrieved context

In [5]:
def ask_groq(question, context):
    groq_api_key = os.getenv('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError('GROQ_API_KEY is not set in the environment or .env file')

    prompt = f'''Answer the question using only the provided context.
If the context does not contain the answer, say you do not know.

Context:
{context}

Question: {question}'''
    request = Request(
        'https://api.groq.com/openai/v1/chat/completions',
        data=json.dumps({
            'model': GROQ_MODEL,
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': 1000,
        }).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {groq_api_key}',
            'Content-Type': 'application/json',
            'User-Agent': 'genai-rag-example/1.0',
        },
        method='POST',
    )
    try:
        with urlopen(request) as response:
            result = json.load(response)
    except HTTPError as error:
        details = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(
            f'Groq request failed with HTTP {error.code}: {details}'
        ) from error
    return result['choices'][0]['message']['content']

In [ ]:
# question = 'Summarize the main points of this PDF.'
question =='Appointments at Army Headquarters'
retrieved_documents = vector_store.similarity_search(question, k=4)
context = '\n\n'.join(document.page_content for document in retrieved_documents)

print(f'Question: {question}\n')
print(f'Answer: {ask_groq(question, context)}')

Question: How much did Microsoft pay to acquire GitHub?

Answer: Microsoft paid **$7.5 billion** to acquire GitHub.
